<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: Training objectives

**[Training objectives](https://learning.nextia-ai.com/courses/deep-learning/m06/training-objectives/)** · Deep Learning and Transformers Explained · Module 6, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** make training examples from plain text, see why a next-token model needs the causal mask, and train a tiny transformer classifier with its shapes annotated.

| | |
|---|---|
| **Time** | About 40 minutes, with the module practice |
| **Needs** | A browser and a free Colab or Kaggle account. The setup cells download the tickets (about 2 MB) and the course's `ticketnet.py`. In Kaggle, turn on Internet in the notebook settings. |
| **You should know** | The causal mask, from [Position and masking](https://learning.nextia-ai.com/courses/deep-learning/m06/position-and-masking/); cross-entropy loss, from [Loss: how wrong is the network?](https://learning.nextia-ai.com/courses/deep-learning/m03/loss/). |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m06/training-objectives/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M06-L03-training-objectives/training-objectives-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Next-token examples from one sentence

The text is its own answer key: each token after the first is the answer for the tokens before it. Run the cell.

In [ ]:
sentence = "My parcel arrived broken, please send a new lamp."
tokens = tokenize(sentence)
print(len(tokens), tokens)
for i in range(1, len(tokens)):
    print(" ".join(tokens[:i]), "->", tokens[i])

> **Check.** You should see 11 tokens and 10 lines, from `my -> parcel` to `... lamp -> .`.

> **Your turn.** Put the number of next-token examples of all validation tickets in `valid_examples`. A ticket of n tokens gives n − 1 examples. Use `tokenize()`. Then run the check cell.

In [ ]:
valid_examples = sum(len(tokenize(row["text"])) - 1 for row in valid_rows)
print(valid_examples)

In [ ]:
expect_hash('valid_examples', valid_examples, '6860985d4ed0adfa')

## 2. Masked-token examples

A masked example hides some tokens anywhere and keeps them as the answers. Run the cell.

In [ ]:
import random

random.seed(3)
hidden = sorted(random.sample(range(len(tokens)), 2))     # hide 2 of the 11 tokens
masked = ["[MASK]" if i in hidden else t for i, t in enumerate(tokens)]
print(" ".join(masked))
print("answers:", {i: tokens[i] for i in hidden})

> **Check.** You should see `[MASK]` in place of "broken" and "lamp".

## 3. Why the causal mask matters in training

The next cell defines a small next-word model. `causal=True` uses the causal mask; `causal=False` lets every position read every token, also its own answer. Run the cell. It prints the shape of the training sequences.

In [ ]:
from torch import nn

vocab = Vocab([row["text"] for row in train_rows])

def sequences(rows, length=48):
    """Each ticket's first 48 token IDs, padded with 0."""
    ids = [vocab.encode(row["text"])[:length] for row in rows]
    return torch.tensor([s + [PAD] * (length - len(s)) for s in ids])

seq_train, seq_valid = sequences(train_rows), sequences(valid_rows)

class NextWordModel(nn.Module):
    def __init__(self, vocab_size, dim=64, length=48):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        self.position = nn.Embedding(length, dim)
        block = nn.TransformerEncoderLayer(dim, 4, 4 * dim, dropout=0.0, batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(block, 2, enable_nested_tensor=False)
        self.out = nn.Linear(dim, vocab_size)

    def forward(self, ids, causal=True):
        n = ids.shape[1]
        h = self.embed(ids) + self.position(torch.arange(n))
        mask = nn.Transformer.generate_square_subsequent_mask(n) if causal else None
        return self.out(self.blocks(h, mask=mask, is_causal=causal))

loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

def next_word_loss(model, ids, causal=True):
    scores = model(ids[:, :-1], causal=causal)      # read tokens 0 .. n-2
    targets = ids[:, 1:]                            # the answers: tokens 1 .. n-1
    return loss_fn(scores.reshape(-1, scores.shape[-1]), targets.reshape(-1))

show_shape("seq_train", seq_train)

> **Check.** You should see `seq_train shape (4696, 48)`.

> **Predict.** Which model will reach the lower training loss: with the mask or without it? Run the cell. It trains both for 3 epochs, as in the lesson. It takes one to three minutes.

In [ ]:
def train_next_word(causal, epochs=3):
    set_seed(0)
    model = NextWordModel(len(vocab))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(seq_train))
        for i in range(0, len(order), 32):
            opt.zero_grad()
            loss = next_word_loss(model, seq_train[order[i:i + 32]], causal=causal)
            loss.backward()
            opt.step()
    return model.eval(), round(loss.item(), 4)

honest, honest_last = train_next_word(causal=True)
cheater, cheater_last = train_next_word(causal=False)
print("training loss, last step: with mask", honest_last, "| without mask", cheater_last)

> **Check.** You should see about 3.41 with the mask and 0.54 without it.

Now measure both as they are used: with the future hidden. Run the cell.

In [ ]:
with torch.no_grad():
    print("with mask, valid loss      ", round(next_word_loss(honest, seq_valid).item(), 3))
    print("without mask, future hidden", round(next_word_loss(cheater, seq_valid, causal=True).item(), 3))

> **Check.** You should see about 3.559 with the mask and 11.345 without it: now the model without the mask has the higher loss. It learned to copy the next token, and there is nothing to copy when the future is hidden.

## 4. Module practice: train a tiny transformer classifier and annotate its shapes

A smaller `TinyTransformer`: 32 numbers per token, 2 heads, 1 block. Read the shapes of the worked example. Run the cell.

In [ ]:
set_seed(0)
small = TinyTransformer(len(vocab), dim=32, heads=2, blocks=1)
small.eval()
ids = token_ids(valid_rows[:4], vocab)
with torch.no_grad():
    h = small.embed(ids) + small.position(torch.arange(ids.shape[1]))
    show_shape("ids", ids)
    show_shape("embedded", h)
    h = small.blocks(h, src_key_padding_mask=ids == PAD)
    show_shape("1 block", h)
    show_shape("scores", small(ids))
print("parameters", count_parameters(small))

> **Check.** You should see (4, 93), (4, 93, 32) twice, (4, 5) and `parameters 108421`.

> **Your turn.** Train `small` for 5 epochs with `train()`, seed 0 and `lr=1e-3`. Then send the first 16 validation tickets through it. Put the shape of their token IDs and the shape of the scores, as tuples, in `task_shapes`. Predict the shapes first. Then run the check cell.

In [ ]:
ids_train, ids_valid = token_ids(train_rows, vocab), token_ids(valid_rows, vocab)
y_train, y_valid = labels(train_rows), labels(valid_rows)
set_seed(0)
small = TinyTransformer(len(vocab), dim=32, heads=2, blocks=1)
history = train(small, ids_train, y_train, ids_valid, y_valid, epochs=5, lr=1e-3)

small.eval()
batch16 = token_ids(valid_rows[:16], vocab)
with torch.no_grad():
    scores = small(batch16)
task_shapes = [tuple(batch16.shape), tuple(scores.shape)]
print(task_shapes)

In [ ]:
expect_hash('task_shapes', task_shapes, 'b9a141ffdd5dc2cc')

> **Check.** After 5 epochs, the validation accuracy should be about 0.86. Write one line for each shape: what does each dimension mean? The model answer is on the lesson page.

## Next

You made training examples with no labels, saw why a next-token model needs the causal mask, and annotated a transformer's shapes. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m06/training-objectives/) has SmolLM2's real next-token probabilities, the cost of pretraining, and the module check that counts toward your certificate. Next, in Module 7, you use a model that someone else pretrained.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Training versus inference](https://learning.nextia-ai.com/courses/deep-learning/m07/training-versus-inference/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m06/training-objectives/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.